# Lesson 02 — Non-Local Means Denoising

## Why This Lesson
NLM is the gold standard of classical denoising. Instead of averaging nearby pixels,
it searches the whole image for similar patches and averages those.
The result is dramatically better than any blur-based approach.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('sample.jpg')

# Add noise
noisy = img.copy().astype(np.float32)
noisy += np.random.normal(0, 25, img.shape)
noisy  = np.clip(noisy, 0, 255).astype(np.uint8)

# Gaussian blur baseline
gaussian = cv2.GaussianBlur(noisy, (7,7), 0)

# Median blur baseline
median   = cv2.medianBlur(noisy, 7)

# Non-Local Means — much better but slower
# h: filter strength (10=light, 20=heavy)
# templateWindowSize: patch size (7 recommended)
# searchWindowSize: search area (21 recommended)
nlm = cv2.fastNlMeansDenoisingColored(
    noisy,
    None,
    h=10,                 # luminance filter strength
    hColor=10,            # color filter strength
    templateWindowSize=7,
    searchWindowSize=21
)

fig, axes = plt.subplots(1, 5, figsize=(26, 5))
for ax, im, t in zip(axes,
    [img, noisy, gaussian, median, nlm],
    ['Original clean', 'Noisy (σ=25)', 'Gaussian blur', 'Median blur', 'NLM (best quality)']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(t); ax.axis('off')
plt.suptitle('NLM preserves sharp edges and fine texture — blurs cannot match this', fontsize=12)
plt.show()

# Zoom into a detail region to see the difference clearly
region = (100, 200, 100, 200)  # y1,y2,x1,x2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, t in zip(axes,
    [noisy, gaussian, nlm],
    ['Noisy — zoomed', 'Gaussian — zoomed', 'NLM — zoomed']):
    crop = im[region[0]:region[1], region[2]:region[3]]
    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.set_title(t); ax.axis('off')
plt.suptitle('Zoom in to see NLM keeps texture that Gaussian destroys', fontsize=12)
plt.show()

## Key Takeaway
NLM = the best classical denoising. It's slow (searches whole image for similar patches)
but produces results where texture is preserved and noise is gone.
Use `h=10` for light noise, `h=20` for heavy noise.